# 🚀 Stack 2.9 - Colab Training Notebook

**Zero-cost training on Google Colab free tier with T4 GPU**

This notebook trains a LoRA adapter for Stack 2.9 on **Qwen2.5-Coder-7B** using a free T4 GPU.

⏱️ **Expected runtime:** 3-5 hours
💾 **VRAM needed:** ~12GB (fits in T4's 15GB)
📦 **Output:** `./training_output/` (on Google Drive)

---

**CRITICAL:** All data saved to **Google Drive** to persist through disconnects.

**Instructions:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Run all cells in order
3. **Allow** Drive access when prompted

---


In [ ]:
# Check GPU
!nvidia-smi

## 1️⃣ Mount Google Drive (REQUIRED for persistence)

Click the link, allow access, copy the auth code, paste it, and press Enter.

**Without Drive mounting, training will be lost if Colab disconnects!**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set up paths on Drive - ALL OUTPUT GOES HERE
import os
BASE_PATH = "/content/drive/MyDrive/stack-2.9"
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)
print(f"\n✅ Working directory: {os.getcwd()}")
print(f"All outputs will be saved to: {BASE_PATH}")
print("\nCurrent folder contents:")
!ls -la

## 2️⃣ Clone Stack 2.9 Repository

In [ ]:
# Clone into Drive if not already there
if not os.path.exists('stack-2.9'):
    !git clone https://github.com/my-ai-stack/stack-2.9.git

os.chdir('stack-2.9')
print(f"Now in: {os.getcwd()}")
!ls -la

## 3️⃣ Install Dependencies

Takes 5-10 minutes.

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.40.0 peft==0.10.0 accelerate bitsandbytes==0.43.0 datasets pyyaml tqdm
!pip install scipy
print("\n✅ Dependencies installed")

## 4️⃣ Download Base Model (Qwen2.5-Coder-7B)

This takes ~15-20 minutes. The model is ~14GB.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B"
MODEL_DIR = "./base_model_qwen7b"

from transformers import AutoModelForCausalLM, AutoTokenizer
import os

if not os.path.exists(MODEL_DIR):
    print(f"Downloading {MODEL_NAME} to {MODEL_DIR}...")
    print("This will take 15-20 minutes...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model.save_pretrained(MODEL_DIR)
    print(f"✅ Model saved to {MODEL_DIR}")
else:
    print(f"✅ Model already exists at {MODEL_DIR}")

!ls -lh {MODEL_DIR} | head -10

## 5️⃣ Prepare Training Data

Using the bundled mini dataset (5K examples) for quick prototyping.

In [ ]:
# Check if data exists in the repo, if not create mini dataset
import os

DATA_PATH = "./data/final/train.jsonl"

if os.path.exists(DATA_PATH):
    print(f"✅ Training data found at {DATA_PATH}")
    !wc -l {DATA_PATH}
else:
    print("Creating mini dataset (5K examples)...")
    !python scripts/create_mini_dataset.py --size 5000 --output data_mini/train_mini.jsonl
    DATA_PATH = "./data_mini/train_mini.jsonl"
    !ls -lh {DATA_PATH}

## 6️⃣ Prepare Training Configuration

In [ ]:
# Use Colab config and update paths
import yaml

config_path = "./stack/training/train_config_local.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update for Colab/T4 GPU
config['model']['name'] = MODEL_DIR
config['data']['input_path'] = DATA_PATH
config['output']['lora_dir'] = "./training_output/lora"
config['output']['merged_dir'] = "./training_output/merged"
config['hardware']['device'] = "cuda"  # Use T4 GPU
config['hardware']['num_gpus'] = 1

# Save updated config
OUTPUT_DIR = "./training_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
updated_config_path = f"{OUTPUT_DIR}/train_config.yaml"

with open(updated_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config saved to: {updated_config_path}")
print("\nConfig summary:")
print(f"  - Model: {config['model']['name']}")
print(f"  - Data: {config['data']['input_path']}")
print(f"  - Device: {config['hardware']['device']}")
print(f"  - Epochs: {config['training']['num_epochs']}")

## 7️⃣ Train LoRA Adapter

⚠️ **This takes 3-5 hours. DO NOT INTERRUPT.**

If Colab disconnects, reconnect and training will resume from checkpoint automatically.

Watch for `Train loss:` decreasing. It should start ~2.0-3.0 and trend downward.

Checkpoints saved every 200 steps to `./training_output/lora/` (on Drive).

In [ ]:
import os
import sys

# Add training module to path
sys.path.insert(0, './stack/training')

print("="*60)
print("STARTING TRAINING")
="*60)
print(f"Working directory: {os.getcwd()}")
print(f"Config: {updated_config_path}")
print(f"Output: {OUTPUT_DIR}/lora")
print("="*60 + "\n")

# Run training
from train_lora import train_lora

trainer = train_lora(updated_config_path)

print("\n" + "="*60)
print("TRAINING FINISHED OR STOPPED")
print("="*60)

## 8️⃣ Verify Training Output

In [ ]:
lora_dir = f"{OUTPUT_DIR}/lora"
print(f"Checking LoRA output: {lora_dir}")

if os.path.exists(lora_dir):
    files = os.listdir(lora_dir)
    print(f"✅ LoRA adapter found! {len(files)} files:")
    for f in files:
        size = os.path.getsize(os.path.join(lora_dir, f)) / (1024*1024)
        print(f"   - {f}: {size:.1f} MB")
else:
    print("⚠️ LoRA directory not found - training may have failed")

## 9️⃣ Merge LoRA Adapter with Base Model

Combines the trained adapter with the base model to produce a standalone fine-tuned model.

In [ ]:
import yaml
import sys

sys.path.insert(0, './stack/training')
from merge_adapter import merge_adapter

merged_dir = f"{OUTPUT_DIR}/merged"

print("="*60)
print("MERGING LORA ADAPTER")
="*60)

# Create merge config
merge_config = {
    'model': {'name': MODEL_DIR, 'trust_remote_code': True},
    'output': {'lora_dir': f'{OUTPUT_DIR}/lora', 'merged_dir': merged_dir},
    'quantization': {'enabled': False}
}

merge_config_path = f"{OUTPUT_DIR}/merge_config.yaml"
with open(merge_config_path, 'w') as f:
    yaml.dump(merge_config, f)

# Run merge
merge_adapter(
    config_path=merge_config_path,
    lora_path=f"{OUTPUT_DIR}/lora",
    output_path=merged_dir
)

print(f"\n✅ Merged model saved to: {merged_dir}")
!ls -lh {merged_dir}/

## 🔟 Test Inference (Quick Check)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = f"{OUTPUT_DIR}/merged"
print(f"Loading model from {model_path}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="cuda",
        trust_remote_code=True
    )
    
    prompt = "Write a Python function to reverse a string:\n\n```python\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    print("Generating...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("="*40)
    print("RESPONSE:")
    print("="*40)
    print(response[len(prompt):])
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nThis is expected if training hasn't completed yet.")

## 🔚 Training Complete!

Your model is ready in `./training_output/merged/` and saved to Google Drive.

**Next steps:**
1. **Download** `training_output/merged/` from Drive to your local machine
2. **Run evaluation**: Evaluate on HumanEval/MBPP benchmarks
3. **Upload model** to Hugging Face Hub
4. **Apply to Together AI**

**Note:** This model was trained on the full/mini dataset. For better performance, consider training on more data or more epochs.
